In [ ]:
import random
import numpy as np

# Dimensiones del tablero de juego (10x10)
DIMENSIONES_TABLERO = 10

# Caracteres para representar diferentes estados en el tablero
CARACTER_AGUA = " "        # Casilla sin explorar o sin barco
CARACTER_BARCO = "O"       # Casilla con un barco
CARACTER_IMPACTO = "X"     # Casilla donde se acertó un barco
CARACTER_FALLO = "-"       # Casilla donde se falló (agua)

# Nombres de los jugadores
NOMBRES_JUGADORES = ["Jugador 1", "Maquina"]

# Variable de control para saber de quién es el turno
TU_TURNO = True

# Diccionario que almacenará información de los barcos
BARCOS = {}

# Bucle para generar dinámicamente los barcos
# barco: tamaño del barco (1 a 4 celdas)
# indice: número secuencial de barcos del mismo tamaño
for barco in range(1, 5):
    # Para cada tamaño, se crean (5-tamaño) barcos
    # Barcos de tamaño 1: 4 barcos
    # Barcos de tamaño 2: 3 barcos
    # Barcos de tamaño 3: 2 barcos
    # Barcos de tamaño 4: 1 barco
    for indice in range(5 - barco):
        # Crea un nombre único para cada barco (ej: "barco1_1", "barco2_1")
        nombre_barco = f"barco{barco}_{indice + 1}"
        # Asigna el tamaño del barco al diccionario
        BARCOS[nombre_barco] = barco


# Definir clase tablero:

class tablero:
    def __init__(self, boats, id_player):
        self.id_player = str(id_player) # Para almacenar el nombre del jugador
        self.boats = boats # Para almacenar el diccionario de los barcos y sus esloras
        self.posiciones = {} # Para almacenar las coordenadas de cada barco
        self.ocupadas = set() # Para almacenar las coordenadas ocupadas
        self.tablero = np.full((10,10), " ") # Para almacenar el tablero con los barcos


# Método de clase para generar coordenadas diferentes para cada barco:

    def generar_coordenadas(self, fila, col, eslora, orientacion):

        coords = [] # Lista con las coordenadas

        for i in range(eslora): # Dependiendo del número de eslora se generan más o menos posiciones

            if orientacion == "N":
                posicion_f, posicion_c = fila - i, col
            elif orientacion == "S":
                posicion_f, posicion_c = fila + i, col
            elif orientacion == "E":
                posicion_f, posicion_c = fila, col + i
            elif orientacion == "O":
                posicion_f, posicion_c = fila, col - i

            coords.append((posicion_f, posicion_c))

        return coords 


# Definir método de clase para colocar barcos:
    def colocar_barcos(self):

        for nombre, eslora in self.boats.items():

            for _ in range(50):  # intentos limitados

                fila = random.randint(0, DIMENSIONES_TABLERO - 1)
                col = random.randint(0, DIMENSIONES_TABLERO - 1)
                orientacion = random.choice(["N", "S", "E", "O"])

                coords = self.generar_coordenadas(fila, col, eslora, orientacion)

                # Validar solapamiento y posición
                valido = True

                for posicion_f, posicion_c in coords:

                    if not (0 <= posicion_f < DIMENSIONES_TABLERO and 0 <= posicion_c < DIMENSIONES_TABLERO):
                        valido = False
                        break

                    if (posicion_f, posicion_c) in self.ocupadas:
                        valido = False
                        break

                # Guardar las posiciones en el diccionario si son válidas
                if valido:

                    self.posiciones[nombre] = coords

                    for posicion_f, posicion_c in coords:
                        self.tablero[posicion_f][posicion_c] = nombre
                        self.ocupadas.add((posicion_f, posicion_c))

                    break

# Método para que el jugador realice un disparo en el tablero del oponente

def realizar_disparo(disparo, tablero):
    acierto = False

    match tablero.tablero[disparo]:
        #Si el disparo acierta un barco, se marca con "X" y se indica que ha sido un acierto
        case "b":
            tablero.tablero[disparo] = "X"
            acierto = True
            print("¡Tocado!")
        #Si el disparo no acierta ningún barco, se marca con "-" y se indica que ha sido un fallo
        case " ":
            tablero.tablero[disparo] = "-"
            print("Agua.")
        #Si el disparo ya se ha realizado en esa posición, se indica que ya se ha disparado allí
        case "X", "-":
            print("Ya has disparado aquí.")
    
    # Se muestra el tablero del oponente después de cada disparo y devuelve si ha sido un acierto o no
    mostrar_tablero_oponente(tablero)
    return acierto

# Método para que la máquina realice un disparo aleatorio en el tablero del jugador

def realizar_disparo_aleatorio(tablero):
    acierto = False
    # La máquina genera coordenadas aleatorias para disparar
    disparo = (random.randint(0, DIMENSIONES_TABLERO - 1), random.randint(0, DIMENSIONES_TABLERO - 1))

    match tablero.tablero[disparo]:
        #Si el disparo acierta un barco, se marca con "X" y se indica que ha sido un acierto
        case "b":
            tablero.tablero[disparo] = "X"
            acierto = True
            print("¡Tocado!")
        #Si el disparo no acierta ningún barco, se marca con "-" y se indica que ha sido un fallo
        case " ":
            tablero.tablero[disparo] = "-"
            print("Agua.")
        #Si el disparo ya se ha realizado en esa posición, se indica que ya se ha disparado allí
        case "X", "-":
            print("Ya has disparado aquí.")

    mostrar_tablero(tablero)
    return acierto

# Se muestra el tablero del jugador con los barcos visibles

def mostrar_tablero(tablero):
    print(tablero.id_player)
    print(tablero.tablero)
    print("******************************************")

# Se muestra el tablero del oponente

def mostrar_tablero_oponente(tablero):
    # Se muestra el tablero del oponente con los barcos ocultos (reemplazando "b" por " ")
    tablero_oculto = np.where(tablero.tablero == "b", " ", tablero.tablero)
    print(tablero.id_player)
    print(tablero_oculto)
    print("******************************************")

num_aciertos1 = 0
num_aciertos2 = 0
tablero1 = tablero(BARCOS, NOMBRES_JUGADORES[0])
tablero2 = tablero(BARCOS, NOMBRES_JUGADORES[1])
tablero1.colocar_barcos()
tablero2.colocar_barcos()
mostrar_tablero(tablero1)
mostrar_tablero_oponente(tablero2)
while num_aciertos1 < 10 and num_aciertos2 < 10:
    if TU_TURNO:
        print("Tu turno:")
        coord_x = int(input("Coordenada x: "))
        coord_y = int(input("Coordenada y: "))
        mostrar_tablero(tablero1)
        TU_TURNO = realizar_disparo((coord_x, coord_y), tablero2)
    else:
        print("Turno de la máquina...")
        TU_TURNO = not realizar_disparo_aleatorio(tablero1)
        mostrar_tablero_oponente(tablero2)

Jugador 1
[[' ' ' ' 'b' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' 'b']
 [' ' 'b' 'b' 'b' ' ' ' ' ' ' ' ' ' ' 'b']
 ['b' ' ' ' ' ' ' ' ' ' ' ' ' ' ' 'b' 'b']
 [' ' ' ' ' ' ' ' ' ' 'b' ' ' ' ' ' ' 'b']
 [' ' ' ' ' ' ' ' ' ' 'b' ' ' ' ' ' ' 'b']
 [' ' ' ' ' ' 'b' ' ' ' ' 'b' 'b' ' ' 'b']
 [' ' ' ' ' ' 'b' ' ' ' ' ' ' 'b' ' ' ' ']
 [' ' ' ' ' ' 'b' ' ' ' ' ' ' ' ' ' ' ' ']]
******************************************
Maquina
[[' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ']]
******************************************
Tu turno:



KeyboardInterrupt

